In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
dfModis = pd.read_parquet(r"data\processed\MODIS\year=2000\data.parquet")
dfViirs = pd.read_parquet(r"data\processed\VIIRS\year=2020\data.parquet")
dfModis.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,country,source_file
0,33.8851,67.6344,313.2,1.0,1.0,2000-11-01,637,Terra,MODIS,55,6.2,297.0,6.7,D,0,Afghanistan,modis_2000_Afghanistan.csv
1,35.4901,66.1055,309.0,1.2,1.1,2000-11-06,655,Terra,MODIS,55,6.2,286.2,10.6,D,0,Afghanistan,modis_2000_Afghanistan.csv
2,33.5200,63.6482,304.3,1.2,1.1,2000-11-17,637,Terra,MODIS,59,6.2,289.2,6.7,D,0,Afghanistan,modis_2000_Afghanistan.csv
3,34.6771,63.4327,300.2,1.3,1.1,2000-12-22,707,Terra,MODIS,0,6.2,285.0,4.7,D,0,Afghanistan,modis_2000_Afghanistan.csv
4,37.4573,66.0029,310.9,3.5,1.8,2000-12-25,559,Terra,MODIS,0,6.2,284.6,68.0,D,0,Afghanistan,modis_2000_Afghanistan.csv


In [4]:
# Eliminar todos aquellos que su tipo sea distinto a 0, ya que suelen ser volcanes, irregularidades
# en el mar o quemadores de gas
dfViirs = dfViirs[dfViirs["type"] == 0]
dfViirs["confidence"].unique()


<ArrowStringArray>
['n', 'h', 'l']
Length: 3, dtype: str

In [5]:
dfViirs.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,country,source_file
0,30.32937,63.24997,351.66,0.34,0.56,2020-01-01,930,N20,VIIRS,n,2,293.01,5.26,D,0,Afghanistan,viirs-jpss1_2020_Afghanistan.csv
1,31.08033,62.00629,341.66,0.39,0.36,2020-01-08,859,N20,VIIRS,n,2,296.25,10.63,D,0,Afghanistan,viirs-jpss1_2020_Afghanistan.csv
2,31.06735,62.00843,341.66,0.39,0.36,2020-01-08,859,N20,VIIRS,n,2,300.56,26.90,D,0,Afghanistan,viirs-jpss1_2020_Afghanistan.csv
3,31.06982,62.00418,330.70,0.39,0.36,2020-01-08,859,N20,VIIRS,n,2,295.92,5.49,D,0,Afghanistan,viirs-jpss1_2020_Afghanistan.csv
4,31.07040,62.00823,337.64,0.39,0.36,2020-01-08,859,N20,VIIRS,n,2,299.69,18.53,D,0,Afghanistan,viirs-jpss1_2020_Afghanistan.csv


In [ ]:
# Armonización, que ambos sensores digan lo mismo
# We don't combine bright cols because they are not equivalent
canonicCols = ["latitude", "longitude", "acq_date", "acq_time", "instrument", "satellite", "confidence",
               "daynight", "country", "frp", "scan", "track", "type", "version"]

years = np.arange(2023, 2026, 1)
sensors = ["MODIS", "VIIRS"]

for year in years:
    yearDfList = []
    for sensor in sensors:
        file_path = Path(fr"data\processed\{sensor}\year={year}\data.parquet")

        if file_path.is_file():
            # Esquema común con columnas canonicas
            df = pd.read_parquet(file_path, columns=canonicCols)
            raw_rows = len(df)

            # Eliminar detecciones de tipo diferentes a 0
            df = df[df["type"] == 0].copy()
            removed = raw_rows - len(df)

            # Armonizar confianza
            if sensor == "MODIS":
                df["confidence"] = np.select(
                    [
                        df["confidence"].between(0, 29),
                        df["confidence"].between(30, 79),
                        df["confidence"].between(80, 100)
                    ], ["l", "n", "h"],
                default=None)

            # Transformar a valores enteros la columna acq_time
            df["acq_time"] = pd.to_numeric(df["acq_time"], errors="coerce").astype("Int64")

            # Crear área de pixel
            df["pixel_area"] = df["scan"] * df["track"]
            df.drop(columns=["scan", "track"], inplace=True)

            print(f"{sensor} {year}: {raw_rows} filas, {removed} descartadas por type != 0")

            yearDfList.append(df)

    # Crear la dirección
    harmonized_path = Path(fr"data\harmonized\year={year}")
    harmonized_path.mkdir(parents=True, exist_ok=True)

    # Save Parquet
    final_df = pd.concat(
        yearDfList,
        ignore_index=True
    )

    final_df.to_parquet(
        harmonized_path / "data.parquet",
        index=False
    )

    print(f"Año {year}: {len(final_df)} detecciones")



C:\Users\santy\AppData\Local\Temp\ipykernel_14216\1827669899.py:38: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["scan", "track"], inplace=True)


MODIS 2023: 4664302 filas, 190698 descartadas por type != 0


C:\Users\santy\AppData\Local\Temp\ipykernel_14216\1827669899.py:38: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["scan", "track"], inplace=True)


VIIRS 2023: 21288380 filas, 1357028 descartadas por type != 0
Año 2023: 24404956 detecciones


C:\Users\santy\AppData\Local\Temp\ipykernel_14216\1827669899.py:38: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["scan", "track"], inplace=True)


MODIS 2024: 4848247 filas, 195116 descartadas por type != 0
